# NUFROST

In [ ]:
import os
import time
import math, argparse, json, hashlib
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Tuple, Sequence, Union, cast

import numpy as np
import rasterio  # type: ignore
try:
    import finufft  # type: ignore
except ModuleNotFoundError as e:
    raise ModuleNotFoundError("finufft is required. Install with: pip install finufft") from e
try:
    from joblib import Parallel, delayed, cpu_count  # type: ignore
    JOBLIB_AVAILABLE = True
except ModuleNotFoundError:
    Parallel = None  # type: ignore
    delayed = None  # type: ignore
    cpu_count = None  # type: ignore
    JOBLIB_AVAILABLE = False
try:
    from tqdm import tqdm  # type: ignore
    TQDM_AVAILABLE = True
except ModuleNotFoundError:
    tqdm = None  # type: ignore
    TQDM_AVAILABLE = False


@dataclass
class Args:
    mount_point: str
    image: str
    cache_dir: str
    force_refresh: bool
    start_time: str
    end_time: str
    target_time: str
    time_unit: str
    modes: int
    eps: float
    num_peaks: int
    power_cum: float
    ignore_dc_hz: float
    refine_peaks: bool
    include_trend: bool
    ridge: float
    freq_weight: float
    huber_iters: int
    huber_delta: float
    min_obs: int
    n_jobs: int
    show_progress: bool
    progress_every: int
    output_path: str

print("Finish")

In [ ]:
def mnt_g_drive(mnt_point: str) -> None:
    from google.colab import drive # type: ignore
    drive.mount(mnt_point)

print("Finish")

In [ ]:
def build_args() -> Args:
    ap = argparse.ArgumentParser()
    ap.add_argument("-g", "--mount-point", type=str, help="folder path to mount Google Drive, e.g. /content/drive",
                    default="/content/drive")
    ap.add_argument("-i", "--image", help="path to input multi-band time-series GeoTIFF", type=str,
                    default="drive/MyDrive/test_cube_0.1_degree/cube_test_B2_0.1.tif")
    ap.add_argument("-c", "--cache-dir", type=str, default="./cache", help="directory for cached npz cubes")
    ap.add_argument("--force-refresh", action="store_true", help="ignore cached npz and rebuild from the tif")

    # Time settings
    ap.add_argument("--start-time", type=str, default="2015-01-01T00:00:00",
                    help="start time in ISO format")
    ap.add_argument("--end-time", type=str, default="2024-01-01T00:00:00",
                    help="end time in ISO format")
    ap.add_argument("--target-time", type=str, default=None,
                    help="target reconstruction time in ISO format")
    ap.add_argument("--time-unit", type=str, default="seconds", choices=("seconds", "days"),
                    help="units used for timestamps")

    # Frequency/fitting settings
    ap.add_argument("--modes", type=int, default=4096)
    ap.add_argument("--eps", type=float, default=1e-12)
    ap.add_argument("--num-peaks", type=int, default=8)
    ap.add_argument("--power-cum", type=float, default=0.7)
    ap.add_argument("--ignore-dc-hz", type=float, default=1e-6)
    ap.add_argument("--refine-peaks", action="store_false", help="disable parabolic peak refinement")
    ap.add_argument("--include-trend", action="store_false", help="disable linear trend in fit")
    ap.add_argument("--ridge", type=float, default=1e-2)
    ap.add_argument("--freq-weight", type=float, default=2.0)
    ap.add_argument("--huber-iters", type=int, default=3)
    ap.add_argument("--huber-delta", type=float, default=1.5)
    ap.add_argument("--min-obs", type=int, default=12, help="minimum valid observations per pixel")
    ap.add_argument("--n-jobs", type=int, default=2,
                    help="number of parallel workers; 0=auto, 1=serial")
    ap.add_argument("--show-progress", action="store_true",
                    help="show progress bar/ETA if available")
    ap.add_argument("--progress-every", type=int, default=50,
                    help="serial mode: print progress every N rows when tqdm is unavailable")

    # Output settings
    ap.add_argument("--output-path", type=str, default="./recon.tif",
                    help="output GeoTIFF path for reconstructed image")

    args = ap.parse_args([])
    if args.target_time is None:
        args.target_time = args.start_time
    return Args(**vars(args))

print("Finish")

In [ ]:
class RSCube:
    """
    Handle spatiotemporal data cube from a multi-band GeoTIFF where each band is a timestamped slice.
    Data are cached as compressed npz to avoid repeatedly reading the large TIFF.
    """
    def __init__(self, tif_path: str, cache_dir: str = "./cache", npz_path: Optional[str] = None, force_refresh: bool = False) -> None:
        self.tif_path = Path(tif_path)
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.npz_path = Path(npz_path) if npz_path else None
        self.force_refresh = force_refresh
        self.meta: Dict[str, object] = {}

    def _file_signature(self) -> str:
        stat = self.tif_path.stat()
        payload = f"{self.tif_path.resolve()}:{stat.st_size}:{stat.st_mtime}"
        return hashlib.md5(payload.encode("utf-8")).hexdigest()[:12]

    def _cache_path(self) -> Path:
        if self.npz_path:
            return self.npz_path
        stem = self.tif_path.stem
        return self.cache_dir / f"{stem}_{self._file_signature()}.npz"

    def _parse_band_timestamp(self, name: str) -> str:
        """Best-effort parse; fall back to the raw band name."""
        tokens = [name.strip()]
        digits = "".join(ch for ch in name if ch.isdigit())
        if len(digits) >= 8:
            tokens.append(digits[:8])
        if len(digits) >= 14:
            tokens.append(digits[:14])
        fmts = ("%Y%m%d", "%Y-%m-%d", "%Y/%m/%d", "%Y%m%dT%H%M%S", "%Y-%m-%dT%H:%M:%S")
        for tok in tokens:
            for fmt in fmts:
                try:
                    return datetime.strptime(tok, fmt).isoformat()
                except ValueError:
                    continue
        return name.strip() or "band"

    def _read_tif(self) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        with rasterio.open(self.tif_path) as src:
            arr = src.read(masked=True).astype(np.float32)
            if np.ma.isMaskedArray(arr):
                arr = arr.filled(np.nan)
            band_names = list(src.descriptions)
            if not band_names or all(name is None or name == "" for name in band_names):
                band_names = [f"band_{i+1}" for i in range(src.count)]
            timestamps = [self._parse_band_timestamp(name) for name in band_names]
            self.meta = {
                "transform": list(src.transform),
                "crs_wkt": src.crs.to_wkt() if src.crs else None,
                "height": src.height,
                "width": src.width,
                "count": src.count,
            }
            return arr, np.array(timestamps, dtype="U32"), np.array(band_names, dtype="U64")

    def _save_npz(self, cache_path: Path, cube: np.ndarray, timestamps: np.ndarray, band_names: np.ndarray) -> None:
        meta = {**self.meta, "cache_path": str(cache_path), "tif_path": str(self.tif_path)}
        np.savez_compressed(cache_path, cube=cube, timestamps=timestamps, band_names=band_names, meta=json.dumps(meta))

    def _load_npz(self, cache_path: Path) -> Dict[str, object]:
        with np.load(cache_path, allow_pickle=False) as z:
            cube = z["cube"]
            timestamps = z["timestamps"]
            band_names = z["band_names"]
            meta = json.loads(z["meta"].item()) if "meta" in z else {}
        self.meta = meta
        return {"cube": cube, "timestamps": timestamps, "band_names": band_names, **meta}

    def load(self) -> Dict[str, object]:
        cache_path = self._cache_path()
        if cache_path.exists() and (not self.force_refresh):
            return self._load_npz(cache_path)
        cube, timestamps, band_names = self._read_tif()
        self._save_npz(cache_path, cube, timestamps, band_names)
        return {"cube": cube, "timestamps": timestamps, "band_names": band_names, **self.meta, "cache_path": str(cache_path)}

print("Finish")


In [ ]:
def _to_seconds_since_start(ts_utc: np.ndarray) -> np.ndarray:
    t0 = np.min(ts_utc)
    return np.ascontiguousarray(ts_utc - t0, dtype=np.float64)


def _parse_timestamp_str(ts: str) -> Optional[datetime]:
    ts = ts.strip()
    fmts = (
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d",
        "%Y%m%dT%H%M%S",
        "%Y%m%d",
    )
    for fmt in fmts:
        try:
            return datetime.strptime(ts, fmt)
        except ValueError:
            continue
    try:
        return datetime.fromisoformat(ts)
    except ValueError:
        return None


def _datetime64_to_seconds(ts: np.datetime64) -> float:
    epoch = np.datetime64("1970-01-01T00:00:00")
    return float((ts - epoch) / np.timedelta64(1, "s"))


def _timestamps_to_seconds(timestamps: np.ndarray, unit: str = "seconds") -> np.ndarray:
    out = []
    for ts in timestamps:
        if isinstance(ts, datetime):
            out.append(ts.timestamp())
            continue
        if isinstance(ts, np.datetime64):
            out.append(_datetime64_to_seconds(ts))
            continue
        dt = _parse_timestamp_str(str(ts))
        if dt is None:
            out.append(np.nan)
            continue
        out.append(dt.timestamp())
    arr = np.array(out, dtype=np.float64)
    if unit == "days":
        arr = arr / 86400.0
    return arr


def next_even(n: int) -> int:
    return int(np.ceil(n/2.0))*2


def refine_parabolic(f: np.ndarray, P: np.ndarray, i: int) -> float:
    if i <= 0 or i >= len(P)-1:
        return float(f[i])
    y0,y1,y2 = P[i-1],P[i],P[i+1]
    denom = (y0 - 2*y1 + y2)
    if denom == 0:
        return float(f[i])
    delta = 0.5*(y0 - y2)/denom
    return float(f[i] + delta*(f[i+1] - f[i]))


def select_peaks_adaptive(f_pos: np.ndarray, P_pos: np.ndarray, k_max: int, power_cum: float, ignore_dc_hz: float, fmax: float) -> np.ndarray:
    lower = max(ignore_dc_hz, 0.0)
    if not np.isfinite(fmax) or fmax <= 0:
        fmax = np.nanmax(f_pos[np.isfinite(f_pos)]) or 1.0
    valid = np.isfinite(f_pos) & np.isfinite(P_pos) & (f_pos > lower) & (f_pos <= fmax)
    if not np.any(valid):
        return np.array([], dtype=int)
    idx = np.where(valid)[0]
    order = np.argsort(-P_pos[idx])
    idx_sorted = idx[order]
    cum = np.cumsum(P_pos[idx_sorted])
    thr = np.clip(power_cum, 0.0, 1.0) * cum[-1]
    take = np.searchsorted(cum, thr) + 1
    take = min(take, k_max, len(idx_sorted))
    return idx_sorted[:take]


def design_matrix(t: np.ndarray, freqs: Union[Sequence[float], np.ndarray], include_trend: bool = True, include_dc: bool = True) -> np.ndarray:
    cols = []
    if include_dc:
        cols.append(np.ones_like(t))
    if include_trend:
        cols.append(t - t.mean())
    for f in freqs:
        w = 2*np.pi*f
        cols.append(np.cos(w*t))
        cols.append(np.sin(w*t))
    return np.vstack(cols).T if cols else np.empty((len(t),0))


def huber_weights(r: np.ndarray, delta: float) -> np.ndarray:
    a = np.abs(r)
    w = np.ones_like(r)
    m = a > delta
    w[m] = (delta / a[m])
    return w


def _safe_lstsq(X: np.ndarray, y: np.ndarray, rcond: Optional[float] = None) -> np.ndarray:
    try:
        return np.linalg.lstsq(X, y, rcond=rcond)[0]
    except Exception:
        try:
            eps = 1e-8
            p = X.shape[1]
            return np.linalg.lstsq(
                np.vstack([X, np.sqrt(eps)*np.eye(p)]),
                np.concatenate([y, np.zeros(p)]),
                rcond=None
            )[0]
        except Exception:
            eps2 = 1e-4
            p = X.shape[1]
            return np.linalg.lstsq(
                np.vstack([X, np.sqrt(eps2)*np.eye(p)]),
                np.concatenate([y, np.zeros(p)]),
                rcond=None
            )[0]


def ridge_with_freq_weights(X: np.ndarray, y: np.ndarray, freqs: Optional[np.ndarray], lam: float, include_dc: bool = True, include_trend: bool = True, freq_weight: float = 2.0, w: Optional[np.ndarray] = None) -> Tuple[np.ndarray, np.ndarray]:
    if X.size == 0:
        return np.zeros(0, dtype=float), np.full_like(y, np.nanmean(y))
    if w is None:
        w = np.ones_like(y)
    W = np.sqrt(w)
    Xw = X * W[:, None]
    yw = y * W

    if lam <= 0:
        beta = _safe_lstsq(Xw, yw)
        y_hat = X @ beta
        return beta, y_hat

    p = X.shape[1]
    R = np.zeros(p, dtype=np.float64)
    col = 0
    if include_dc:
        col += 1
    if include_trend:
        col += 1
    if freqs is not None and len(freqs) > 0:
        fmax_local = np.max(freqs) if np.max(freqs) > 0 else 1.0
        if not np.isfinite(fmax_local) or fmax_local <= 0:
            fmax_local = 1.0
        for f in freqs:
            w_f = (max(f, 0.0) / fmax_local) ** max(0.0, freq_weight)
            if col < p:
                R[col] = w_f
                col += 1
            if col < p:
                R[col] = w_f
                col += 1

    if np.any(R > 0):
        X_aug = np.vstack([Xw, np.diag(np.sqrt(lam)*R)])
    else:
        X_aug = np.vstack([Xw, np.sqrt(lam)*np.eye(p)])
    y_aug = np.concatenate([yw, np.zeros(p, dtype=np.float64)])

    beta = _safe_lstsq(X_aug, y_aug)
    y_hat = X @ beta
    return beta, y_hat


def robust_fit_freq_ridge(X: np.ndarray, y: np.ndarray, freqs: np.ndarray, lam: float, iters: int, delta: float, include_dc: bool, include_trend: bool, freq_weight: float) -> Tuple[np.ndarray, np.ndarray]:
    if X.shape[1] == 0:
        return np.zeros(0), np.full_like(y, np.nanmean(y))
    w = np.ones_like(y)
    y_hat = np.zeros_like(y)
    for _ in range(max(0, iters)):
        r = y - y_hat
        w = huber_weights(r, max(1e-8, delta))
        _, y_hat = ridge_with_freq_weights(X, y, freqs, lam,
                                           include_dc, include_trend, freq_weight, w=w)
    beta, y_hat = ridge_with_freq_weights(X, y, freqs, lam,
                                          include_dc, include_trend, freq_weight, w=w)
    return beta, y_hat


# =============== Single-pixel fit and predict ===============

def predict_single_pixel(t_sec: np.ndarray, y: np.ndarray, target_t: float,
                         nufft_modes: int, eps: float,
                         num_peaks: int, power_cum: float, ignore_dc_hz: float,
                         refine_peaks: bool, include_trend: bool,
                         ridge_lam: float, freq_weight: float, huber_iters: int, huber_delta: float,
                         min_obs: int) -> Tuple[float, int]:
    m = np.isfinite(y) & np.isfinite(t_sec)
    if m.sum() < max(3, min_obs):
        return np.nan, 0
    t = np.asarray(t_sec[m], dtype=np.float64)
    yy = np.asarray(y[m], dtype=np.float64)

    t_rel = _to_seconds_since_start(t)
    t_rel_mean = float(t_rel.mean())
    Tspan = float(t_rel.max() - t_rel.min())
    if not np.isfinite(Tspan) or Tspan <= 0:
        return np.nan, 0

    x = 2*np.pi*(t_rel - t_rel.min())/Tspan - np.pi
    x = np.ascontiguousarray(x, dtype=np.float64)
    c = np.ascontiguousarray(yy.astype(np.complex128))
    ms = next_even(nufft_modes)
    Fk = finufft.nufft1d1(x, c, ms, eps=eps, isign=-1)
    k = np.arange(-ms//2, ms//2, dtype=np.int64)
    freqs = k.astype(np.float64)/Tspan

    pos = freqs >= 0
    f_pos = freqs[pos]
    P_pos = (np.abs(Fk[pos])**2)

    dt = np.diff(np.sort(t_rel))
    dt_pos = dt[dt > 0]
    dt_med = float(np.median(dt_pos)) if dt_pos.size else Tspan/len(t_rel)
    fmax = 0.5 / max(dt_med, 1e-12)

    pos_idx = select_peaks_adaptive(f_pos, P_pos, k_max=num_peaks,
                                    power_cum=power_cum,
                                    ignore_dc_hz=ignore_dc_hz, fmax=fmax)
    if len(pos_idx) == 0:
        freqs_sel = np.array([], dtype=np.float64)
    else:
        if refine_peaks:
            freqs_sel = np.array([refine_parabolic(f_pos, P_pos, i) for i in pos_idx], dtype=np.float64)
        else:
            freqs_sel = np.array(f_pos[pos_idx], dtype=np.float64)

    X = design_matrix(t_rel, freqs_sel, include_trend=include_trend, include_dc=True)
    beta, _ = robust_fit_freq_ridge(X, yy, freqs_sel,
                                    lam=ridge_lam, iters=huber_iters, delta=huber_delta,
                                    include_dc=True, include_trend=include_trend, freq_weight=freq_weight)

    t_star_rel = float(target_t - t.min())
    cols = [1.0]
    if include_trend:
        cols.append(t_star_rel - t_rel_mean)
    for f in freqs_sel:
        w = 2*np.pi*f
        cols.append(math.cos(w*t_star_rel))
        cols.append(math.sin(w*t_star_rel))
    X_star = np.array(cols, dtype=np.float64).reshape(1, -1) if len(cols) else np.zeros((1,0))

    if X.shape[1] > 0:
        y_star = float((X_star @ beta).item())
    else:
        y_star = float(np.nanmean(yy))
    return y_star, len(freqs_sel)


# =============== Cube reconstruction ===============

def revive(cube: np.ndarray, timestamps: np.ndarray, target_time: str, args: Args) -> np.ndarray:
    t_sec = _timestamps_to_seconds(timestamps, unit=args.time_unit)
    target_dt = _parse_timestamp_str(target_time)
    if target_dt is None:
        raise ValueError(f"Unrecognized target_time: {target_time}")
    target_t = target_dt.timestamp()
    if args.time_unit == "days":
        target_t = target_t / 86400.0

    bands, H, W = cube.shape
    out = np.full((H, W), np.nan, dtype=np.float32)

    def _predict_row(i: int) -> Tuple[int, np.ndarray]:
        row = np.full(W, np.nan, dtype=np.float32)
        for j in range(W):
            y = cube[:, i, j]
            pred, _ = predict_single_pixel(
                t_sec, y, target_t,
                args.modes, args.eps,
                args.num_peaks, args.power_cum, args.ignore_dc_hz,
                args.refine_peaks, args.include_trend,
                args.ridge, args.freq_weight, args.huber_iters, args.huber_delta,
                args.min_obs
            )
            row[j] = pred
        return i, row

    n_jobs = args.n_jobs
    if n_jobs == 0:
        if cpu_count is not None:
            n_jobs = max(1, int(cpu_count()))
        else:
            n_jobs = max(1, int(os.cpu_count() or 1))
    n_jobs = min(n_jobs, H)

    if not JOBLIB_AVAILABLE or n_jobs == 1:
        if (not JOBLIB_AVAILABLE) and n_jobs != 1:
            print("joblib not installed; falling back to serial execution. Install with: pip install joblib")
        progress_every = max(1, int(args.progress_every))
        start = time.perf_counter()
        if args.show_progress and TQDM_AVAILABLE and tqdm is not None:
            for i in tqdm(range(H), total=H, desc="Rows", unit="row"):
                _, row = _predict_row(i)
                out[i, :] = row
        else:
            for i in range(H):
                _, row = _predict_row(i)
                out[i, :] = row
                if args.show_progress and (i + 1) % progress_every == 0:
                    elapsed = time.perf_counter() - start
                    rate = (i + 1) / max(elapsed, 1e-9)
                    remaining = (H - (i + 1)) / max(rate, 1e-9)
                    print(f"Rows {i+1}/{H} | ETA ~ {remaining/60:.1f} min")
        return out

    assert Parallel is not None and delayed is not None
    if args.show_progress and TQDM_AVAILABLE and tqdm is not None:
        try:
            results_iter = Parallel(
                n_jobs=n_jobs,
                prefer="processes",
                max_nbytes="256M",
                mmap_mode="r",
                return_as="generator",
            )(delayed(_predict_row)(i) for i in range(H))
            results_iter = cast(Sequence[Tuple[int, np.ndarray]], results_iter)
            for i, row in tqdm(results_iter, total=H, desc="Rows", unit="row"):
                out[i, :] = row
            return out
        except TypeError:
            print("joblib return_as not supported; progress will be approximate.")

    results = Parallel(
        n_jobs=n_jobs,
        prefer="processes",
        max_nbytes="256M",
        mmap_mode="r",
    )(delayed(_predict_row)(i) for i in range(H))
    results = cast(Sequence[Tuple[int, np.ndarray]], results)
    for i, row in results:
        out[i, :] = row
    return out

print("Finish")


In [ ]:
# Example: mount Drive (if provided), then load cube and cache to npz
args = build_args()
if args.mount_point:
    mnt_g_drive(args.mount_point)

cube_loader = RSCube(args.image, cache_dir=args.cache_dir, force_refresh=args.force_refresh)
cube_data = cube_loader.load()
cube = cast(np.ndarray, cube_data["cube"])
timestamps = cast(np.ndarray, cube_data["timestamps"])
print(f"cube shape: {cube.shape}")
print(f"timestamps (first 5): {timestamps[:5]}")
print(f"cache saved at: {cube_data.get('cache_path', 'unknown')}")

# Reconstruct at target time
recon = revive(cube, timestamps, args.target_time, args)
print(f"reconstructed map shape: {recon.shape}")

# Save reconstructed image
output_path = Path(args.output_path)
output_path.parent.mkdir(parents=True, exist_ok=True)
transform = None
if "transform" in cube_data:
    try:
        transform = rasterio.Affine(*cube_data["transform"])
except Exception:
        transform = None
with rasterio.open(
    output_path,
    "w",
    driver="GTiff",
    height=recon.shape[0],
    width=recon.shape[1],
    count=1,
    dtype=recon.dtype,
    crs=cube_data.get("crs_wkt", None),
    transform=transform,
) as dst:
    dst.write(recon, 1)
print(f"saved to: {output_path}")

print("Finish")